#

<div align=center>
<img src="https://uol.unifor.br/acesso/app/autenticacao/assets/img/logos/icon-unifor.svg" width=45 height=45>

<br><br>
<font size=5 color='black'><strong>MBA Ciência de dados:</strong> Estatística descritiva

<strong>Projeto:</strong> Titanic

<strong>Autoria:</strong> Heitor Teixeira

</div>

## <font color=darkblue> 1 - Imports e declaração de constantes

### <font color=steelblue> 1.1 - Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
import numpy as np

### <font color=steelblue> 1.2 - Constantes

Declarar constantes em uma unica celula facilita a manutenção de notebooks longos.<br>Alguns benefícios:
1. Melhorar a legibilidade do codigo
2. As constantes podem ser esquema de cores para padronizar sempre as mesmas cores para determinadas classes, paths de arquivos e datasets e etc.
3. posso modifica-las apenas aqui e servir para o codigo inteiro


In [ ]:
# dataset do professor
PATH_ELEICOES    = "datasets/eleicoes.csv"

# arquivos auxiliares do TSE para enriquecimento
PATH_CANDIDATOS  = "datasets/extras/consulta_cand_2014_BRASIL.csv"
PATH_DESPESAS    = "datasets/extras/despesas_candidatos_2014_brasil.txt"

PATH_DATASET_ENRIQUECIDO = "datasets/extras/eleicoes_enriquecido.csv"

# paleta de cores para deixar os graficos padronizados
CORES = [
    "#4878D0",
    "#EE854A",
    "#6ACC64",
    "#D65F5F",
    "#956CB4",
]

# tamanho padrao dos graficos
FIGSIZE = (10, 5)

# mapeamento de colunas: dataset original do professor
MAP_COLUMNS_ELEICOES = {
    "State":            "Estado",
    "Candidate Number": "Numero Candidato",
    "Money (R$ Reais)": "Gasto",
    "Votes":            "Votos",
}

# mapeamento de colunas: prestacao de contas TSE (despesas)
MAP_COLUMNS_DESPESAS = {
    "UF":               "Estado",
    "Número candidato": "Numero Candidato",
    "Nome candidato":   "Nome",
    "Sigla Partido":    "Partido",
    "CPF do candidato": "CPF",
}

# mapeamento de colunas: cadastro oficial de candidatos TSE
MAP_COLUMNS_CANDIDATOS = {
    "SG_UF":             "Estado",
    "NR_CANDIDATO":      "Numero Candidato",
    "DS_SIT_TOT_TURNO":  "Situacao",
    "DS_GENERO":         "Genero",
    "DS_GRAU_INSTRUCAO": "Escolaridade",
    "DS_ESTADO_CIVIL":   "Estado Civil",
    "DS_COR_RACA":       "Raca",
    "DT_NASCIMENTO":     "Dt Nascimento",
}

## <font color=darkblue> 2 - Carregamento e Preparação do Dataset

### <font color=steelblue> 2.1 - Carregando o dataset original

**Observações:**
- o dataset original fornecido pelo professor contém dados de candidatos a Deputado Federal nas eleições de 2014
- cada linha representa um candidato, identificado por Estado e Número de Candidato
- as colunas originais foram renomeadas para português via `MAP_COLUMNS_ELEICOES`
- o csv tem espaco nos valores. quando o valor é numerico não tem problema, python reconhece. mas em string é preciso dar o strip().


In [3]:
eleicoes_df = pd.read_csv(PATH_ELEICOES, header=0, names=MAP_COLUMNS_ELEICOES.values())

# strip() na caoluna str
eleicoes_df["Estado"] = eleicoes_df["Estado"].str.strip()


### <font color=steelblue> 2.2 - Feature Engineering

**Observações:**

O enriquecimento dos dados não foi solicitado, mas reflete uma postura natural do cientista de dados: ir além do dataset fornecido para extrair análises com maior contexto e relevância. Dados isolados, sem um objetivo bem definido dizem pouco. Os dados adicionais dão direção e profundidade para a análise. 

- enriqueci o dataset original com dois arquivos públicos do TSE: prestação de contas de despesas e cadastro oficial de candidatos
- os três arquivos são lidos uma única vez e todos os joins são feitos nesta seção, evitando releituras ao longo do notebook
- a chave de join utilizada é `Estado` + `Numero Candidato` em ambos os merges
- de `despesas_candidatos_2014_brasil.txt`: extrai nome, partido, cargo e CPF por candidato (via `drop_duplicates`) e agrega quantidade de notas e total declarado ao TSE (via `groupby`)
- de `consulta_cand_2014_BRASIL.csv`: filtrei apenas candidatos com número de 4 dígitos, padrão exclusivo de Deputado Federal, e extraí gênero, escolaridade, raça, estado civil e resultado eleitoral

#### <font color=slategray> 2.2.1 - Aquisição de dados externos

In [ ]:
# essa celula é pra rodar só uma vez pra gerar o dataset enriquecido e depois comentar.

despesas_df = pd.read_csv(
    PATH_DESPESAS,
    sep=";",
    encoding="latin1",
    decimal=",",
    usecols=["UF", "Número candidato", "Nome candidato", "Sigla Partido", "Cargo", "CPF do candidato", "Valor despesa"],
    dtype={"CPF do candidato": str},
)

cand_df = pd.read_csv(
    PATH_CANDIDATOS,
    sep=";",
    encoding="latin1",
    usecols=["SG_UF", "NR_CANDIDATO"] + list(MAP_COLUMNS_CANDIDATOS.keys()),
)
''

'\n# essa celula é pra rodar só uma vez para gerar o dataset enriquecido e depois comentar.\n\ndespesas_df = pd.read_csv(\n    PATH_DESPESAS,\n    sep=";",\n    encoding="latin1",\n    decimal=",",\n    usecols=["UF", "Número candidato", "Nome candidato", "Sigla Partido", "Cargo", "CPF do candidato", "Valor despesa"],\n    dtype={"CPF do candidato": str},\n)\n\ncand_df = pd.read_csv(\n    PATH_CANDIDATOS,\n    sep=";",\n    encoding="latin1",\n    usecols=["SG_UF", "NR_CANDIDATO"] + list(MAP_COLUMNS_CANDIDATOS.keys()),\n)\n'

#### <font color=slategray> 2.2.2 - Agregações e merges

In [ ]:

qtd_notas = (
    despesas_df
    .groupby(["UF", "Número candidato"])
    .agg(Qtd_Notas=("Valor despesa", "count"), Total_Despesas_TSE=("Valor despesa", "sum"))
    .reset_index()
    .rename(columns=MAP_COLUMNS_DESPESAS)
)

candidatos_despesas = (
    despesas_df
    .drop_duplicates(subset=["UF", "Número candidato"])
    .rename(columns=MAP_COLUMNS_DESPESAS)
    .merge(qtd_notas, on=["Estado", "Numero Candidato"], how="left")
    [["Estado", "Numero Candidato", "Nome", "Partido", "Cargo", "CPF", "Qtd_Notas", "Total_Despesas_TSE"]]
)

candidatos = (
    cand_df[cand_df["NR_CANDIDATO"].astype(str).str.len() == 4]
    .drop_duplicates(subset=["SG_UF", "NR_CANDIDATO"])
    .rename(columns=MAP_COLUMNS_CANDIDATOS)
)

eleicoes_enriquecido = (
    eleicoes_df
    .merge(candidatos_despesas, on=["Estado", "Numero Candidato"], how="left")
    .merge(candidatos,          on=["Estado", "Numero Candidato"], how="left")
)

eleicoes_enriquecido["Qtd_Notas"] = eleicoes_enriquecido["Qtd_Notas"].fillna(0).astype(int)

eleicoes_enriquecido.to_csv(PATH_DATASET_ENRIQUECIDO, index=False)


#### <font color=slategray> 2.2.3 - Dataset enriquecido

In [6]:
eleicoes_enriquecido = pd.read_csv(PATH_DATASET_ENRIQUECIDO)

print(f"Tabela 2.2 - Dataset enriquecido ({len(eleicoes_enriquecido)} candidatos)")
display(eleicoes_enriquecido.head(5).set_index("Numero Candidato"))

Tabela 2.2 - Dataset enriquecido (6353 candidatos)


,Estado,Gasto,Votos,Nome,Partido,Cargo,CPF,Qtd_Notas,Total_Despesas_TSE,Dt Nascimento,Genero,Escolaridade,Estado Civil,Raca,Situacao
Numero Candidato,,,,,,,,,,,,,,,
1919,AC,35504.34,515,JUAREZ PEDROSA CAVALCANTE,PTN,Deputado Federal,1.333425e+10,30,35504.34,24/02/1961,MASCULINO,ENSINO MÉDIO COMPLETO,DIVORCIADO(A),PARDA,SUPLENTE
1212,AC,397136.76,11397,JOSÉ LUIS SCHAFER,PDT,Deputado Federal,3.142027e+10,223,397136.76,20/06/1960,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,SUPLENTE
5012,AC,1580.00,15,ANA PAULA MORAIS DE HOLANDA,PSOL,Deputado Federal,8.214671e+10,3,1580.00,10/11/1985,FEMININO,ENSINO MÉDIO COMPLETO,SOLTEIRO(A),PARDA,NÃO ELEITO
1321,AC,66093.33,1913,ROSELI COSTA,PT,Deputado Federal,3.080045e+10,21,66093.33,04/06/1970,FEMININO,SUPERIOR COMPLETO,CASADO(A),PARDA,SUPLENTE
1144,AC,296327.21,13610,VANDA DENIR MILANI NOGUEIRA,PP,Deputado Federal,7.858185e+10,178,296327.21,17/09/1953,FEMININO,SUPERIOR COMPLETO,CASADO(A),BRANCA,SUPLENTE
